# 05 — Evaluation

Evaluate LightGBM on the held-out **test set** (last 15% of data, strictly chronological).

**Metrics:** Accuracy, Precision, Recall, F1, ROC-AUC  
**Plots:** Confusion matrices, ROC curves, calibration curves, feature importance  
**Final output:** Summary table — horizon × metric


In [4]:
import sys
from pathlib import Path

# Resolve project root robustly (works in VS Code Jupyter where CWD = workspace root)
try:
    ROOT = Path(__vsc_ipynb_file__).parent.parent.resolve()
except NameError:
    ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.features import build_features, get_feature_cols
from src.models import LightGBMModel
from src.evaluate import (
    chronological_split,
    evaluate_model,
    plot_confusion_matrix,
    plot_roc_curves,
    plot_calibration,
    aggregate_results,
    print_results_table,
)

sns.set_theme(style='darkgrid', palette='muted')

PROCESSED  = ROOT / 'data' / 'processed'
MODEL_DIR  = ROOT / 'models'
HORIZONS   = ['5m', '30m', '1h']


## 1. Prepare Test Sets

In [ ]:
feat_frames = {}
flat_splits = {}
splits = {}

for label in HORIZONS:
    raw = pd.read_csv(PROCESSED / f'btc_{label}.csv',
                      index_col='timestamp', parse_dates=True)
    df = build_features(raw)
    feat_frames[label] = df

    train_df, val_df, test_df = chronological_split(df)
    splits[label] = (train_df, val_df, test_df)

    feat_cols = get_feature_cols(df)
    X_flat = df[feat_cols].values.astype(np.float32)
    y_flat = df['target'].values.astype(np.float32)
    n = len(y_flat)
    tr = int(n * 0.70)
    vl = int(n * 0.85)
    flat_splits[label] = (
        (X_flat[:tr],    y_flat[:tr]),
        (X_flat[tr:vl],  y_flat[tr:vl]),
        (X_flat[vl:],    y_flat[vl:]),
    )


## 2. Load Saved Models

In [ ]:
lgbm_models = {}

for label in HORIZONS:
    m = LightGBMModel()
    m.load(MODEL_DIR / f'lgbm_{label}.pkl')
    lgbm_models[label] = m

print('All models loaded.')


## 3. Run Evaluation on Test Sets

In [ ]:
all_results = {}  # {horizon: {model_name: metrics + raw preds}}

for label in HORIZONS:
    (_, _), (_, _), (X_te, y_te) = flat_splits[label]

    print(f'\n=== {label} ===')
    res = evaluate_model(lgbm_models[label], X_te, y_te)
    res['y_true'] = y_te
    all_results[label] = {'lgbm': res}
    print(f"  lgbm  acc={res['accuracy']:.4f}  "
          f"f1={res['f1']:.4f}  auc={res['roc_auc']:.4f}")


## 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(12, 4))

for ax, label in zip(axes, HORIZONS):
    res = all_results[label]['lgbm']
    plot_confusion_matrix(
        res['y_true'], res['y_pred'],
        title=f'LGBM — {label}',
        ax=ax,
    )

plt.tight_layout()
plt.show()


## 5. ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, label in zip(axes, HORIZONS):
    plot_roc_curves(
        all_results[label],
        ax=ax,
        title=f'ROC Curves — {label}',
    )

plt.tight_layout()
plt.show()

## 6. Calibration Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, label in zip(axes, HORIZONS):
    plot_calibration(
        all_results[label],
        ax=ax,
        title=f'Calibration — {label}',
    )

plt.tight_layout()
plt.show()

## 7. Prediction Confidence Distribution

In [ ]:
label = '1h'
res = all_results[label]['lgbm']
probs = res['y_prob']
true  = res['y_true'].astype(int)

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(probs[true == 0], bins=30, alpha=0.6, color='tomato',
        density=True, label='True DOWN')
ax.hist(probs[true == 1], bins=30, alpha=0.6, color='mediumseagreen',
        density=True, label='True UP')
ax.axvline(0.5, color='black', linewidth=0.8, linestyle='--')
ax.set_title(f'LGBM — {label} prediction confidence')
ax.set_xlabel('P(UP)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 8. Final Summary Table

In [ ]:
results_df = aggregate_results(all_results)
print_results_table(results_df)

In [ ]:
# Heatmap of ROC-AUC across model × horizon
auc_pivot = results_df['roc_auc'].unstack(level='horizon')[HORIZONS]

fig, ax = plt.subplots(figsize=(6, 3))
sns.heatmap(
    auc_pivot, annot=True, fmt='.4f', cmap='YlGn',
    vmin=0.48, vmax=0.65, linewidths=0.5, ax=ax,
)
ax.set_title('ROC-AUC by Model & Horizon (Test Set)')
ax.set_xlabel('Horizon')
ax.set_ylabel('Model')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of Accuracy
acc_pivot = results_df['accuracy'].unstack(level='horizon')[HORIZONS]

fig, ax = plt.subplots(figsize=(6, 3))
sns.heatmap(
    acc_pivot, annot=True, fmt='.4f', cmap='Blues',
    vmin=0.48, vmax=0.60, linewidths=0.5, ax=ax,
)
ax.set_title('Accuracy by Model & Horizon (Test Set)')
ax.set_xlabel('Horizon')
ax.set_ylabel('Model')
plt.tight_layout()
plt.show()

## 9. Live Inference Demo

To run live inference with the LightGBM model:
```bash
# LightGBM on 5-minute candles
python -m src.live \\
    --model-path models/lgbm_5m.pkl \\
    --interval 5m

# LightGBM on 1-hour candles
python -m src.live \\
    --model-path models/lgbm_1h.pkl \\
    --interval 1h
```

Expected output every 60 s:
```
[2025-01-15 14:32:00 UTC]  BTCUSDT 5m  → UP  ▲  confidence=58.3%  candle=45.2% complete  last_price=94320.50
```
